# MyoSuite Finger Pose Tutorial

This notebook allows you to run the custom musculoskeletal finger environment training and visualization directly in Google Colab.

### Step 1: Clone the Repository & Install Dependencies
This cell will clone the project's GitHub repository, navigate into the project directory, and install all required dependencies using `uv` and the `pyproject.toml` file.

In [ ]:
%cd /content

!rm -rf biomech-rl-tutorial
!git clone https://github.com/MichalMiazga/biomech-rl-tutorial.git
%cd biomech-rl-tutorial
!pip install uv
!uv pip install --system .

### Step 2: Train the Custom Reward Environment
Now, let's train the agent using our custom reward function and muscle noise.

In [ ]:
import os
import myosuite
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from gymnasium.wrappers import TimeLimit
from reward_tutorial import CustomRewardPoseEnv

def make_custom_env(**kwargs):
    env = CustomRewardPoseEnv(**kwargs)
    return TimeLimit(env, max_episode_steps=100)


# Use myosuite.__file__ to get the exact path to the package directory
myosuite_dir = os.path.dirname(myosuite.__file__)
model_path = os.path.join(myosuite_dir, "simhive", "myo_sim", "finger", "myofinger_v0.xml")

In [ ]:
# --- Main Configuration ---
PRINT_REWARD_COMPONENTS = True # Set to True to see the reward components and use DummyVecEnv

# Define the arguments for the environment, which will be passed to each parallel process
env_kwargs = {
    "model_path": model_path,
    "target_jnt_range": {
        "IFadb": (0, 0),
        "IFmcp": (0, 0),
        "IFpip": (0.75, 0.75),
        "IFdip": (0.75, 0.75),
    },
    "viz_site_targets": ("IFtip",),
    "normalize_act": True,
    "use_muscle_noise": True,
    "print_reward_components": PRINT_REWARD_COMPONENTS,
    "custom_reward_weights": {
        "pose": 1.0,
        "bonus": 4.0,
        "act_reg": 1.0,
        "penalty": 50,
    }
}

# Automatically switch between fast parallel training and slow debug training
if PRINT_REWARD_COMPONENTS:
    print("Debug mode: Using DummyVecEnv to show reward components.")
    vec_env_class = DummyVecEnv
    n_envs = 1
else:
    print("Performance mode: Using SubprocVecEnv for fast parallel training.")
    vec_env_class = SubprocVecEnv
    n_envs = 10

env = make_vec_env(
    make_custom_env,
    n_envs=n_envs,
    vec_env_cls=vec_env_class,
    env_kwargs=env_kwargs
)

# Save a checkpoint every 10000 steps
checkpoint_callback = CheckpointCallback(save_freq=10000, save_path='./logs/',
                                         name_prefix='myo_finger_custom_reward')

model = PPO("MlpPolicy", env, verbose=1)

print(f"Starting training on {n_envs} custom reward environment(s) using {vec_env_class.__name__}...")
# Train for 100_000 timesteps.
model.learn(total_timesteps=100_000, callback=checkpoint_callback)

# Save the final model
os.makedirs('models', exist_ok=True)
model.save("models/ppo_myofinger_custom_reward")
print("Training finished and model saved to 'models/ppo_myofinger_custom_reward.zip'.")


### Step 3: Visualize the Results
Run the visualization script to generate the MP4 video of the trained agent.

In [ ]:
!python visualize.py

### Step 4: Display the Video in Colab
We can use `moviepy` and IPython display tools to watch the generated video right here in the notebook!

In [ ]:
from moviepy.editor import VideoFileClip
from IPython.display import HTML

# Path to the generated video
video_path = 'video/episode_custom_reward.mp4'

# Display the video
HTML(f"""
<video width="640" height="480" controls>
  <source src="{video_path}" type="video/mp4">
</video>
""")